#CELL 1
# Module 2.2 — SHAP Explainability

This notebook explains the selected Logistic Regression attrition model using SHAP.

SHAP values quantify how much each transformed feature moved an individual employee's
predicted attrition risk above or below the model's baseline expectation.

In [ ]:
#CELL 2
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import shap

from src.analytics.insights import (
    compute_shap_values,
    get_global_feature_importance,
    get_top_factors_for_employee,
)
from src.data.load import load_raw_ibm_hr
from src.data.preprocess import clean_hr_data
from src.modeling.attrition import (
    load_attrition_model,
    prepare_attrition_data,
    split_attrition_data,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURES_DIR = ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
#CELL 3
raw_df = load_raw_ibm_hr()
clean_df = clean_hr_data(raw_df)

X, y = prepare_attrition_data(clean_df)
X_train, X_test, y_train, y_test = split_attrition_data(X, y)

model = load_attrition_model()

print(f"Training employees: {len(X_train):,}")
print(f"Test employees: {len(X_test):,}")
print(f"Attrition rate - test set: {y_test.mean():.1%}")

In [ ]:
#CELL 4
shap_values, transformed_X = compute_shap_values(
    model=model,
    X_background=X_train.head(200),
    X_explain=X_test.head(200),
)

print(f"SHAP values shape: {shap_values.values.shape}")
print(f"Transformed feature matrix shape: {transformed_X.shape}")

In [ ]:
#CELL 5
global_importance = get_global_feature_importance(
    shap_values=shap_values,
    top_n=20,
)

global_importance

In [ ]:
#CELL 6
fig, ax = plt.subplots(figsize=(10, 7))

top_features = global_importance.head(15).iloc[::-1]

ax.barh(
    top_features["feature"],
    top_features["mean_absolute_shap_value"],
    color="#2563eb",
)

ax.set_xlabel("Mean absolute SHAP value")
ax.set_ylabel("Transformed feature")
ax.set_title("Global SHAP feature importance for attrition prediction")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_global_importance.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#CELL 7
shap.summary_plot(
    shap_values,
    transformed_X,
    max_display=20,
    show=False,
)

plt.gcf().set_size_inches(10, 7)
plt.title("SHAP summary plot", pad=20)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_summary_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#CELL 8
employee_position = 0

top_factors = get_top_factors_for_employee(
    shap_values=shap_values,
    transformed_features=transformed_X,
    employee_position=employee_position,
    top_n=10,
)

top_factors

In [ ]:
#CELL 9
shap.plots.waterfall(
    shap_values[employee_position],
    max_display=10,
    show=False,
)

plt.gcf().set_size_inches(10, 7)
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "shap_employee_waterfall.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
#CELL 10
shap.initjs()

shap.plots.force(
    shap_values[employee_position],
    matplotlib=True,
    show=False,
)

plt.gcf().set_size_inches(14, 4)
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "shap_employee_force_plot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

#CELL 11
## Interpretation

- Positive SHAP values increase the predicted probability of attrition.
- Negative SHAP values decrease the predicted probability of attrition.
- The global importance chart shows which transformed features have the largest average impact across the sample.
- The waterfall plot explains one employee's prediction by showing the features that pushed their risk above or below the model baseline.
- These explanations must be treated as model explanations, not proof of causation.

In [ ]:
#CELL 12
global_importance.to_csv(
    ROOT / "reports" / "shap_global_feature_importance.csv",
    index=False,
)

top_factors.to_csv(
    ROOT / "reports" / "shap_employee_top_factors.csv",
    index=False,
)

print("Saved:")
print(ROOT / "reports" / "shap_global_feature_importance.csv")
print(ROOT / "reports" / "shap_employee_top_factors.csv")